# Topic: SQL Median Calculation

## Definition (30-second explanation)
* The median is the precise middle value of a dataset sorted in ascending or descending order.
* If the dataset has an odd number of values, it is the exact middle row. 
* If the dataset has an even number of values, the median is the average of the two middle rows.
* Most SQL dialects lack a built-in `MEDIAN()` aggregate function, making it a test of advanced query skills.

## Why Interviewers Ask This
* It is one of the most popular SQL interview questions at top companies like Amazon, Google, Meta, and Stripe.
* Tests your mastery of window functions (like `ROW_NUMBER()`) and self-joins or CTEs.
* Evaluates your attention to edge cases like even/odd row counts, integer division truncation, and `NULL` handling.

## Core Concepts
* **`PERCENTILE_CONT(0.5)`**: A clean, readable window function available in PostgreSQL, SQL Server, Oracle, and Snowflake that automatically handles interpolation.
* **Universal `ROW_NUMBER()` Method**: Using ascending and descending row numbers to logically isolate the middle row(s) in dialects like MySQL.
* **The `ABS` Trick**: Filtering where `ABS(rn_asc - rn_desc) <= 1` elegantly captures both odd (difference of 0) and even (difference of 1) middle rows.

## When to Use
* Essential for analyzing skewed distributions where outliers distort the mean.
* Standard metric for income/salary distribution, real estate pricing, and customer lifetime value.

## Advantages
* Highly robust to extreme outliers (e.g., one CEO salary won't skew the metric for standard employees).
* Provides a more accurate reflection of the "typical" or "average" user/case in skewed data.

## Limitations
* Computationally expensive because calculating the median always requires fully sorting the dataset.
* Cumbersome syntax in standard SQL compared to simple aggregations like `AVG()` or `SUM()`.

## Common Comparisons
* **Mean vs. Median**: Mean (`AVG()`) is heavily influenced by outliers, while Median is robust.
* **`PERCENTILE_CONT` vs. `PERCENTILE_DISC`**: `CONT` mathematically interpolates the exact median for even datasets, whereas `DISC` simply returns an existing actual data point from the set.

## Common Interview Traps
* **Ignoring NULLs**: NULLs can shift row counts; always use `WHERE column IS NOT NULL`.
* **Integer Truncation**: In some dialects, dividing an integer count by 2 truncates the decimal; use `total/2.0` or cast to float.
* **Ignoring Grouping Context**: Failing to clarify if the interviewer wants the median for the entire dataset or grouped by a specific category.

## Python / SQL Syntax
**Method 1: Modern SQL (PostgreSQL, Snowflake, etc.)**
```sql
SELECT department,
       PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY salary) AS median_salary
FROM employees
GROUP BY department;
```

**Method 2: Universal Approach (MySQL, etc.)**
```sql
WITH numbered AS (
  SELECT department, salary,
         ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary ASC) AS rn_asc,
         ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) AS rn_desc
  FROM employees
  WHERE salary IS NOT NULL
)
SELECT department, AVG(salary) AS median_salary
FROM numbered
WHERE ABS(rn_asc - rn_desc) <= 1
GROUP BY department;
```

## 45-Second Interview Answer
"Because standard SQL doesn't have a built-in MEDIAN() function, I would calculate it based on the database engine. If we are using PostgreSQL or Snowflake, I would use the PERCENTILE_CONT(0.5) WITHIN GROUP function because it cleanly handles even and odd counts. If I need a universal solution for something like MySQL, I would use a CTE to assign ascending and descending ROW_NUMBER()s partitioned by the group, filter out NULL values, and then average the rows where the absolute difference between the ascending and descending row numbers is less than or equal to 1."

## Exxample Questions:

### Q1: Calculate the median order value per customer category for the last 6 months.

* **Ideal Interview Answer:** Because MySQL doesn't have a median function, I would use a CTE to assign ascending and descending row numbers to the order values partitioned by category. I would filter for the last 6 months first, then average the middle rows where the absolute difference between the row numbers is 1 or 0.
```sql
    WITH NumberedOrders AS (
        SELECT customer_category, 
               order_value,
               ROW_NUMBER() OVER (PARTITION BY customer_category ORDER BY order_value ASC) AS rn_asc,
               ROW_NUMBER() OVER (PARTITION BY customer_category ORDER BY order_value DESC) AS rn_desc
        FROM orders
        WHERE order_date >= CURRENT_DATE - INTERVAL 6 MONTH
          AND order_value IS NOT NULL
    )
    SELECT customer_category, 
           AVG(order_value) AS median_order_value
    FROM NumberedOrders
    WHERE ABS(rn_asc - rn_desc) <= 1
    GROUP BY customer_category;
```
* **Common Mistakes:** Forgetting the date filter before calculating the median, or calculating the median over the whole dataset instead of partitioning by customer category.
* **Likely Interviewer Follow-up:** How would you optimize this query if the orders table contains 10 billion rows and sorting is too slow? (Answer: Suggest using approximate quantile functions or pre-aggregating data into buckets, depending on the exact engine capabilities).

### Q2: Find all products where the price is above the category median price.

* **Ideal Interview Answer:** I'd use two CTEs. The first assigns the ascending and descending row numbers to product prices per category. The second calculates the median by averaging the middle rows. Finally, I'd join this median CTE back to the main products table to filter for prices above the median.
```sql
    WITH CategoryNumbered AS (
        SELECT category_id, 
               price,
               ROW_NUMBER() OVER (PARTITION BY category_id ORDER BY price ASC) AS rn_asc,
               ROW_NUMBER() OVER (PARTITION BY category_id ORDER BY price DESC) AS rn_desc
        FROM products
        WHERE price IS NOT NULL
    ),
    CategoryMedians AS (
        SELECT category_id, 
               AVG(price) AS median_price
        FROM CategoryNumbered
        WHERE ABS(rn_asc - rn_desc) <= 1
        GROUP BY category_id
    )
    SELECT p.product_id, p.product_name, p.price, c.median_price
    FROM products p
    JOIN CategoryMedians c ON p.category_id = c.category_id
    WHERE p.price > c.median_price;
```
* **Common Mistakes:** Attempting to filter by the window function directly in the `WHERE` clause without using a CTE or subquery.
* **Likely Interviewer Follow-up:** What if multiple products are exactly equal to the median price? How does your query handle them? (Answer: Strictly using `>` excludes them; we'd need to confirm with stakeholders if it should be `>=`).

### Q3: Compare the mean vs median salary in each department. In which departments does the mean exceed the median significantly?

* **Ideal Interview Answer:** I would calculate the mean salary as a window function in the first CTE while also assigning the row numbers. In the final query, I would group by department, average the middle rows for the median, and select the max of the mean salary, allowing me to easily calculate the difference.
```sql
    WITH DepartmentStats AS (
        SELECT department, 
               salary,
               AVG(salary) OVER (PARTITION BY department) AS mean_salary,
               ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary ASC) AS rn_asc,
               ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) AS rn_desc
        FROM employees
        WHERE salary IS NOT NULL
    ),
    MedianStats AS (
        SELECT department,
               MAX(mean_salary) AS mean_salary,
               AVG(salary) AS median_salary
        FROM DepartmentStats
        WHERE ABS(rn_asc - rn_desc) <= 1
        GROUP BY department
    )
    SELECT department, 
           mean_salary, 
           median_salary,
           (mean_salary - median_salary) AS skew_difference
    FROM MedianStats
    ORDER BY skew_difference DESC;
```
* **Common Mistakes:** Calculating the average *after* filtering for the middle rows, which would result in the mean being identical to the median. The mean must be calculated on the full dataset before the `ABS(rn_asc - rn_desc) <= 1` filter is applied.
* **Likely Interviewer Follow-up:** If the mean is significantly higher than the median, what does that tell you about the data distribution? (Answer: It indicates a right-skewed distribution, meaning a few very high salaries are pulling the average up).

### Q4: Find the median number of support tickets resolved per day for each support agent.

* **Ideal Interview Answer:** This requires a multi-step aggregation. First, a CTE to count tickets per agent per day. Second, a CTE to assign the row numbers to those daily counts partitioned by agent. Finally, average the middle rows from the second CTE.
```sql
    WITH DailyCounts AS (
        SELECT agent_id, 
               DATE(resolved_at) AS resolution_date,
               COUNT(ticket_id) AS tickets_resolved
        FROM support_tickets
        GROUP BY agent_id, DATE(resolved_at)
    ),
    NumberedCounts AS (
        SELECT agent_id, 
               tickets_resolved,
               ROW_NUMBER() OVER (PARTITION BY agent_id ORDER BY tickets_resolved ASC) AS rn_asc,
               ROW_NUMBER() OVER (PARTITION BY agent_id ORDER BY tickets_resolved DESC) AS rn_desc
        FROM DailyCounts
    )
    SELECT agent_id, 
           AVG(tickets_resolved) AS median_daily_tickets
    FROM NumberedCounts
    WHERE ABS(rn_asc - rn_desc) <= 1
    GROUP BY agent_id;
```
* **Common Mistakes:** Trying to assign row numbers to the raw tickets table without doing the initial daily `COUNT()` aggregation first.
* **Likely Interviewer Follow-up:** How would you handle days where an agent was scheduled but resolved zero tickets? (Answer: The initial `GROUP BY` drops days with zero rows. We would need to `LEFT JOIN` a date dimension table and the agent schedule to ensure zero-count days are explicitly generated before calculating the median).

## Practice Questions:

### Q1:
**Scenario:**
You are interviewing for a Data Scientist role. The interviewer states: "We use a legacy MySQL database that does not support the PERCENTILE_CONT window function. We need to analyze our credit risk."

**Your Task:**
Write a SQL query to calculate the median credit limit for customers in each country. You must handle edge cases like even/odd counts and ignore customers who have a NULL credit limit.

**Mock Schema**
```sql
customerNumber,customerName,country,creditLimit
103,Atelier graphique,France,21000.00
112,Signal Gift Stores,USA,71800.00
114,Australian Collectors,Australia,117300.00
119,La Rochelle Gifts,France,118200.00
```


* **Answer:** I can solve this using two different CTE approaches. 
  
  The most efficient way is to calculate the row number (sorted ascending) and the total count of rows per partition. Then, I can filter where the row number falls between `total_rows / 2.0` and `total_rows / 2.0 + 1` to capture either the single middle row or the two middle rows, and average them.

    Option 1:
    ```sql
    WITH sorted_creds AS (
        SELECT country, creditLimit,
               ROW_NUMBER() OVER(PARTITION BY country ORDER BY creditLimit) AS rnk,
               COUNT(*) OVER(PARTITION BY country) AS total_rows
        FROM customers
        WHERE creditLimit IS NOT NULL
    )
    SELECT country,
           ROUND(AVG(creditLimit), 2) AS median_creditLimit
    FROM sorted_creds
    WHERE rnk BETWEEN (total_rows / 2.0) AND (total_rows / 2.0 + 1)
    GROUP BY country;
    ```

    Option 2:
    ```sql
    with sorted_creds as (
        select country, creditLimit,
        row_number() over(partition by country order by creditLimit asc) as rnk_asc,
        row_number() over(partition by country order by creditLimit desc) as rnk_desc
    from customers
    where creditLimit is not null
    )
    select country,
	    round(avg(creditLimit),2) as median_creditLimit
    from sorted_creds
    where abs(cast(rnk_asc as signed) - cast(rnk_desc as signed)) <= 1
    group by country;
    ```
    
* **Common Mistakes:** Forgetting that MySQL `ROW_NUMBER` subtraction throws an unsigned out-of-range error if not cast properly. Using integer division instead of float division (`2.0`) in the `BETWEEN` method.
* **Likely Interviewer Follow-up:** Which of these two methods is more computationally efficient and why? (Answer: The first method with `COUNT(*)` is more efficient because it only requires the database engine to sort the partitions once, whereas the ascending/descending method requires sorting the data twice).